# D76 基线5: BC 行为克隆闭环——`base_bc_eval`

热启动 ckpt 直接 greedy 闭环评估(几分钟), 与主方法同 val 口径。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 评估

In [ ]:
import json, torch
from derbysim.learn.features import Featurizer
from derbysim.learn.panel import PanelAdapter
from derbysim.learn.panel_models import PanelOptionPolicy
from derbysim.learn.ppo import TorchPanelDriver
from derbysim.learn.rollout import as_factory, evaluate_policy
from derbysim.learn.scenarios import build_day_trains, day_horizon, load_assets, operating_days
from derbysim.sim.reward import RewardedDynamicEnv
net,params,dwell,ptc=load_assets()
ck=torch.load('artifacts/dataset/bc_fullpanel_gnn.pt',map_location='cpu',weights_only=False)
ad=PanelAdapter(net,Featurizer(net,ptc))   # BC = fv1 口径
pol=PanelOptionPolicy(ck.get('encoder','gnn'),*ck['dims'],hidden=64)
pol.load_state_dict(ck['state_dict']); pol.eval()
envf=lambda: RewardedDynamicEnv(net,params,ptc,dwell,proactive=True,reserve_window='wide',control_classes={'1','2'})
eps=[(str(w['start'])[:16],(lambda w=w: build_day_trains(w)),day_horizon(w)) for w in operating_days('val')]
ev=evaluate_policy(envf,ad,as_factory(TorchPanelDriver(pol,torch.device('cpu'),greedy=True)),eps)
ev.pop('eval_episodes',None)
print(f"BC 闭环(val): 完成 {ev['eval_completion_rate']:.4f} | 延误 {ev['eval_weighted_secondary_delay_s']:.0f} | reward {ev['eval_episode_reward']:.1f}")
json.dump(ev,open('artifacts/reports/base_bc_eval.json','w'))
print('saved -> artifacts/reports/base_bc_eval.json')
